# AutoScout24 Audi Q4 e-tron — Italy Secondary Scraper

Reads the Italy primary CSV from `outputs/`, visits each `listing_url` individually,
parses the detail-page `__NEXT_DATA__` payload, and writes a merged
`outputs/scrape_audi_q4_it_YYYYMMDD_secondary.csv`.

Run this notebook **after** `01_primary_scraper_italy.ipynb`.

In [1]:
from __future__ import annotations

import json
import re
import time
import urllib.error
import urllib.request
from http.cookiejar import CookieJar
from pathlib import Path
from datetime import datetime
import pandas as pd

# Read from the Italy outputs folder
OUTPUTS_DIR = Path("outputs")
FILE_GLOB = "scrape_audi_q4_it_*.csv"

_primary_csvs = sorted(
    [p for p in OUTPUTS_DIR.glob(FILE_GLOB) if not p.stem.endswith("_secondary")],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
INPUT_CSV_PATH = _primary_csvs[0] if _primary_csvs else None

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_DELAY_SECONDS = 0.6
MAX_RETRIES = 3
MAX_LISTINGS = None  # Set to 5 while testing.

USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

NEXT_DATA_PATTERN = re.compile(
    r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>',
    flags=re.DOTALL,
)

SECONDARY_COLUMNS = [
    "listing_url",
    "warranty_exists",
    "warranty_text",
    "has_full_service_history",
    "had_accident",
    "damage_conditions",
    "previous_owner_count",
    "body_type",
    "door_count",
    "seat_count",
    "exterior_color",
    "paint_type",
    "interior_color",
    "upholstery_material",
    "battery_ownership",
    "battery_charging_time",
    "electric_range",
    "electric_range_city",
    "secondary_scrape_status",
    "secondary_scrape_error",
]

print(f"Input CSV: {INPUT_CSV_PATH}")

Input CSV: outputs/scrape_audi_q4_it_20260522.csv


In [2]:
def clean_text(value):
    if value is None:
        return None
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text or None


def format_nested_value(value):
    if value is None:
        return None
    if isinstance(value, dict):
        for key in ("formatted", "label", "value", "description"):
            candidate = clean_text(value.get(key))
            if candidate is not None:
                return candidate
        raw_value = value.get("raw")
        if raw_value is not None:
            return clean_text(raw_value)
        compact = {k: v for k, v in value.items() if v not in (None, "", [], {})}
        return json.dumps(compact, ensure_ascii=False) if compact else None
    if isinstance(value, list):
        parts = [format_nested_value(item) for item in value]
        parts = [p for p in parts if p not in (None, "")]
        return "; ".join(str(p) for p in parts) if parts else None
    if isinstance(value, (bool, int, float)):
        return value
    return clean_text(value)


def build_http_opener():
    cookie_jar = CookieJar()
    opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(cookie_jar))
    opener.addheaders = [
        ("User-Agent", USER_AGENT),
        ("Accept-Language", "en-GB,en;q=0.9"),
    ]
    return opener


def build_output_csv_path(input_csv_path):
    input_csv_path = Path(input_csv_path)
    return input_csv_path.with_name(f"{input_csv_path.stem}_secondary.csv")


def fetch_html(url, opener):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with opener.open(url, timeout=REQUEST_TIMEOUT_SECONDS) as response:
                return response.read().decode("utf-8", errors="ignore")
        except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(REQUEST_DELAY_SECONDS * attempt)
    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def extract_next_data(html):
    match = NEXT_DATA_PATTERN.search(html)
    if not match:
        raise ValueError("Could not find __NEXT_DATA__ on the detail page.")
    return json.loads(match.group(1))


def extract_secondary_fields(listing_details):
    vehicle = listing_details.get("vehicle") or {}
    return {
        "warranty_exists": listing_details.get("warrantyExists"),
        "warranty_text": format_nested_value(listing_details.get("warranty")),
        "has_full_service_history": vehicle.get("hasFullServiceHistory"),
        "had_accident": vehicle.get("hadAccident"),
        "damage_conditions": format_nested_value(vehicle.get("damageConditions")),
        "previous_owner_count": vehicle.get("noOfPreviousOwners"),
        "body_type": format_nested_value(vehicle.get("bodyType")),
        "door_count": vehicle.get("numberOfDoors"),
        "seat_count": vehicle.get("numberOfSeats"),
        "exterior_color": format_nested_value(vehicle.get("bodyColor")),
        "paint_type": format_nested_value(vehicle.get("paintType")),
        "interior_color": format_nested_value(vehicle.get("upholsteryColor")),
        "upholstery_material": format_nested_value(vehicle.get("upholstery")),
        "battery_ownership": format_nested_value(vehicle.get("batteryOwnershipType")),
        "battery_charging_time": format_nested_value(vehicle.get("batteryChargingTime")),
        "electric_range": format_nested_value(vehicle.get("electricRangeWithFallback")),
        "electric_range_city": format_nested_value(vehicle.get("electricRangeCity")),
    }

In [3]:
def fetch_secondary_details(listing_url, opener):
    record = {column: None for column in SECONDARY_COLUMNS}
    record["listing_url"] = clean_text(listing_url)
    try:
        html = fetch_html(listing_url, opener)
        next_data = extract_next_data(html)
        listing_details = next_data["props"]["pageProps"]["listingDetails"]
        record.update(extract_secondary_fields(listing_details))
        record["secondary_scrape_status"] = "ok"
    except Exception as exc:
        record["secondary_scrape_status"] = "error"
        record["secondary_scrape_error"] = clean_text(exc)
    return record


def run_secondary_scrape(input_csv_path):
    input_df = pd.read_csv(input_csv_path)
    if "listing_url" not in input_df.columns:
        raise ValueError("The input CSV must contain a listing_url column.")

    if MAX_LISTINGS is not None:
        input_df = input_df.head(MAX_LISTINGS).copy()

    unique_urls = (
        input_df["listing_url"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    opener = build_http_opener()
    secondary_records = []

    for index, listing_url in enumerate(unique_urls, start=1):
        print(f"[{index}/{len(unique_urls)}] Fetching: {listing_url}")
        secondary_records.append(fetch_secondary_details(listing_url, opener))
        time.sleep(REQUEST_DELAY_SECONDS)

    secondary_df = pd.DataFrame(secondary_records).reindex(columns=SECONDARY_COLUMNS)
    secondary_df = secondary_df.where(pd.notna(secondary_df), None)

    merged_df = input_df.merge(secondary_df, on="listing_url", how="left")
    output_csv_path = build_output_csv_path(input_csv_path)
    merged_df.to_csv(output_csv_path, index=False, encoding="utf-8")

    return input_df, secondary_df, merged_df, output_csv_path

In [4]:
if INPUT_CSV_PATH is None:
    raise FileNotFoundError(
        "No Italy primary CSV found in outputs/. "
        "Run 01_primary_scraper_italy.ipynb first."
    )

output_csv_path = build_output_csv_path(INPUT_CSV_PATH)
print(f"Input CSV:  {INPUT_CSV_PATH}")
print(f"Output CSV: {output_csv_path}")

input_df, secondary_df, merged_df, output_csv_path = run_secondary_scrape(INPUT_CSV_PATH)

print(f"\nFetched secondary details for {len(secondary_df)} unique Italy listings.")
print(f"Merged file saved to: {output_csv_path}")
print(f"\nError rate: {(secondary_df['secondary_scrape_status'] == 'error').mean():.1%}")
secondary_df.head()

Input CSV:  outputs/scrape_audi_q4_it_20260522.csv
Output CSV: outputs/scrape_audi_q4_it_20260522_secondary.csv
[1/227] Fetching: https://www.autoscout24.com/offers/audi-q4-e-tron-45-s-line-edition-quattro-265cv-electric-black-8d7209f6-394c-454f-a474-b673e6502204


[2/227] Fetching: https://www.autoscout24.com/offers/audi-q4-e-tron-q4-sportback-e-tron-40-s-line-edition-electric-white-384baec8-934a-47d7-ae1c-656829b6e5d5


[3/227] Fetching: https://www.autoscout24.com/offers/audi-q4-e-tron-35-business-advanced-electric-grey-48e267d1-5179-43cb-8876-0b527f813493


[4/227] Fetching: https://www.autoscout24.com/offers/audi-q4-e-tron-q4-sportback-e-tron-35-electric-grey-45ae23cf-7af8-43d1-94bc-3ea564b09546


[5/227] Fetching: https://www.autoscout24.com/offers/audi-q4-e-tron-sportback-e-tron-45-business-electric-grey-eccb50be-99d8-4dd3-b390-f68b83f0b638


...

[225/227] Fetching: https://www.autoscout24.com/offers/audi-q4-e-tron-q4-spb-45-e-tron-s-line-edition-electric-silver-cd13c232-9d0f-473e-a4ab-405ab3a1b498


[226/227] Fetching: https://www.autoscout24.com/offers/audi-q4-e-tron-q4-spb-55-e-tron-quattro-business-advanced-electric-silver-69997b59-5e7e-44ae-9c20-ec5a256877e7


[227/227] Fetching: https://www.autoscout24.com/offers/audi-q4-e-tron-q4-spb-55-e-tron-quattro-s-line-edition-electric-silver-75572842-6470-441c-803a-3ee013fa561e



Fetched secondary details for 227 unique Italy listings.
Merged file saved to: outputs/scrape_audi_q4_it_20260522_secondary.csv

Error rate: 0.0%


,listing_url,warranty_exists,warranty_text,has_full_service_history,had_accident,damage_conditions,previous_owner_count,body_type,door_count,seat_count,exterior_color,paint_type,interior_color,upholstery_material,battery_ownership,battery_charging_time,electric_range,electric_range_city,secondary_scrape_status,secondary_scrape_error
0,https://www.autoscout24.com/offers/audi-q4-e-t...,True,24 months,False,False,None,1.0,SUV/Off-Road/Pick-Up,5.0,5.0,Black,Metallic,Black,Alcantara,None,None,None,None,ok,None
1,https://www.autoscout24.com/offers/audi-q4-e-t...,True,24 months,False,False,None,1.0,SUV/Off-Road/Pick-Up,5.0,5.0,White,Others,Black,Other,None,None,None,None,ok,None
2,https://www.autoscout24.com/offers/audi-q4-e-t...,True,48 months,False,False,None,1.0,SUV/Off-Road/Pick-Up,5.0,5.0,Grey,Others,Black,Other,None,None,None,None,ok,None
3,https://www.autoscout24.com/offers/audi-q4-e-t...,True,12 months,True,False,None,NaN,SUV/Off-Road/Pick-Up,5.0,5.0,Grey,Others,Grey,Cloth,None,None,None,None,ok,None
4,https://www.autoscout24.com/offers/audi-q4-e-t...,False,None,False,False,None,1.0,Coupe,5.0,5.0,Grey,None,None,None,None,None,494 km,None,ok,None
